In [3]:
# LDA, PCA and Kernel PCA - Regression HandsOn

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


# 1. Split the dataset
def split_scalar(indep_X, dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(
        indep_X,
        dep_Y,
        test_size=0.25,
        random_state=0
    )

    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    return X_train, X_test, y_train, y_test


# 2. Apply dimensionality-reduction techniques
def LDA_Method(indep_X, dep_Y, n):
    X_train, X_test, y_train, y_test = split_scalar(indep_X, dep_Y)

    lda = LDA(n_components=2)
    X_train = lda.fit_transform(X_train, y_train)
    X_test = lda.transform(X_test)

    return X_train, X_test, y_train, y_test


def PCA_Method(indep_X, dep_Y, n):
    X_train, X_test, y_train, y_test = split_scalar(indep_X, dep_Y)

    pca = PCA(n_components=2)

    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    return X_train, X_test, y_train, y_test


def Kernel_PCA(indep_X, dep_Y, n):
    X_train, X_test, y_train, y_test = split_scalar(indep_X, dep_Y)

    kpca = KernelPCA(
        n_components=2,
        kernel="rbf",
        eigen_solver="arpack"
    )

    X_train = kpca.fit_transform(X_train)
    X_test = kpca.transform(X_test)

    return X_train, X_test, y_train, y_test


# 3. Regression model evaluation using R2 score
def r2_prediction(regressor, X_test, y_test):
    y_pred = regressor.predict(X_test)
    return r2_score(y_test, y_pred)


def Linear(X_train, y_train, X_test, y_test):
    regressor = LinearRegression()
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)


def svm_linear(X_train, y_train, X_test, y_test):
    regressor = SVR(kernel="linear")
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)


def Decision(X_train, y_train, X_test, y_test):
    regressor = DecisionTreeRegressor(random_state=0)
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)


def random(X_train, y_train, X_test, y_test):
    regressor = RandomForestRegressor(
        n_estimators=10,
        random_state=0
    )
    regressor.fit(X_train, y_train)
    return r2_prediction(regressor, X_test, y_test)


# 4. Create the comparison table
def lda_pca_regression(acclin, accsvml, accdes, accrf):
    result_dataframe = pd.DataFrame(
        index=["LDA", "PCA", "Kernel PCA"],
        columns=["Linear", "SVMl", "Decision", "Random"]
    )

    for number, method in enumerate(result_dataframe.index):
        result_dataframe.loc[method, "Linear"] = acclin[number]
        result_dataframe.loc[method, "SVMl"] = accsvml[number]
        result_dataframe.loc[method, "Decision"] = accdes[number]
        result_dataframe.loc[method, "Random"] = accrf[number]

    return result_dataframe.astype(float)


# 5. Import and prepare the dataset
dataset1 = pd.read_csv("Wine.csv", index_col=None)

df2 = dataset1.copy()
df2 = pd.get_dummies(df2, drop_first=True)

target_column = "Customer_Segment"

indep_X = df2.drop(target_column, axis=1)
dep_Y = df2[target_column]


# 6. Apply LDA, PCA and Kernel PCA

X_train_lda, X_test_lda, y_train_lda, y_test_lda = LDA_Method(
    indep_X, dep_Y, 5
)

X_train_pca, X_test_pca, y_train_pca, y_test_pca = PCA_Method(
    indep_X, dep_Y, 5
)

X_train_kpca, X_test_kpca, y_train_kpca, y_test_kpca = Kernel_PCA(
    indep_X, dep_Y, 5
)


# 7. Evaluate all regression models for each technique
reduced_data = [
    (X_train_lda, X_test_lda, y_train_lda, y_test_lda),
    (X_train_pca, X_test_pca, y_train_pca, y_test_pca),
    (X_train_kpca, X_test_kpca, y_train_kpca, y_test_kpca)
]

acclin = []
accsvml = []
accdes = []
accrf = []

for X_train, X_test, y_train, y_test in reduced_data:

    r2_lin = Linear(X_train, y_train, X_test, y_test)
    acclin.append(r2_lin)

    r2_sl = svm_linear(X_train, y_train, X_test, y_test)
    accsvml.append(r2_sl)

    r2_d = Decision(X_train, y_train, X_test, y_test)
    accdes.append(r2_d)

    r2_r = random(X_train, y_train, X_test, y_test)
    accrf.append(r2_r)


# 8. Display the final comparison table
result = lda_pca_regression(acclin, accsvml, accdes, accrf)

print("\nLDA, PCA and Kernel PCA - Regression R2 Comparison\n")
print(result.round(4))



LDA, PCA and Kernel PCA - Regression R2 Comparison

            Linear    SVMl  Decision  Random
LDA         0.8044  0.8034    0.9114  0.9092
PCA         0.7289  0.7255    0.9557  0.9500
Kernel PCA  0.8670  0.8657    0.9114  0.9885
